In [1]:
from pathlib import Path
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Trỏ thẳng vào thư mục data bên trong project
BASE_DATA_DIR = PROJECT_ROOT / "data" / "Livestock_Dataset"

# --- CHỌN BỘ DỮ LIỆU ĐỂ TRAIN TẠI ĐÂY -

DATA_DIR = BASE_DATA_DIR / "Poultry_Feces"

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print(f"Đang nạp dữ liệu từ: {DATA_DIR.name}")
print("Thư mục Data tồn tại:", DATA_DIR.exists())
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

TRAIN_MODE = True

Project root: C:\Users\MangOS\livestock-diseases-ai
Đang nạp dữ liệu từ: Poultry_Feces
Thư mục Data tồn tại: True
Torch: 2.5.1+cu121


Device: cuda


In [2]:
from src.utils import set_seed
from src.dataset import load_dataset, make_loaders
from src.model import build_model, count_parameters, unfreeze_backbone, load_checkpoint
from src.train import train_model
from src.evaluate import run_full_evaluation, plot_training_curves

set_seed(42)
print("Project modules imported successfully.")

Project modules imported successfully.


In [3]:
# Tự động đọc data và weights từ dataset.py mới
train_dataset, val_dataset, test_dataset, info = load_dataset(DATA_DIR)

train_loader, val_loader, test_loader = make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32, # Giữ ở mức 32 để tránh tràn RAM
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

print("Dataset loaded successfully.")
print("Split sizes:", info["split_sizes"])
print("Number of classes:", info["n_classes"])
print("Trọng số phân lớp (Class Weights):", info["class_weights"])

Dataset loaded successfully.
Split sizes: {'train': 5644, 'val': 1206, 'test': 1217, 'total': 8067}
Number of classes: 4
Trọng số phân lớp (Class Weights): tensor([0.8142, 0.8394, 3.5903, 0.7681])


In [4]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Min pixel value:", images.min().item())
print("Max pixel value:", images.max().item())
print("First 10 labels:", labels[:10].tolist())

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Min pixel value: -2.1179039478302
Max pixel value: 2.640000104904175
First 10 labels: [1, 0, 2, 3, 0, 3, 0, 1, 0, 2]


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tự động scale số lớp theo n_classes của dataset
model = build_model(
    n_classes=info["n_classes"],
    freeze_backbone=True,
).to(device)

params = count_parameters(model)

print("Device:", device)
print("Model device:", next(model.parameters()).device)
print("Model created successfully.")
print("Parameters:", params)

Device: cuda
Model device: cuda:0
Model created successfully.
Parameters: {'total': 11178564, 'trainable': 2052, 'frozen': 11176512}


In [6]:
with torch.no_grad():
    sample_outputs = model(images.to(device))

print("Sample output shape:", sample_outputs.shape)

Sample output shape: torch.Size([32, 4])


In [7]:
phase1_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=10,
    learning_rate=0.001,
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_chicken_phase1_best.pth"),
    history_path=str(RESULTS_DIR / "history_chicken_phase1.json"),
    phase_name="phase1_frozen_backbone",
)

print("Phase 1 training completed.")


Training phase1_frozen_backbone
Epochs: 10
Learning rate: 0.001
Trainable parameters: 2,052


train:   0%|                                           | 0/177 [00:00<?, ?it/s]

train:   1%|▏                                  | 1/177 [00:00<01:12,  2.44it/s]

train:   2%|▌                                  | 3/177 [00:00<00:25,  6.80it/s]

train:   3%|▉                                  | 5/177 [00:00<00:16, 10.16it/s]

...

eval:  74%|██████████████████████████▌         | 28/38 [00:01<00:00, 20.00it/s]

eval:  82%|█████████████████████████████▎      | 31/38 [00:01<00:00, 20.05it/s]

eval:  89%|████████████████████████████████▏   | 34/38 [00:01<00:00, 20.16it/s]

eval:  97%|███████████████████████████████████ | 37/38 [00:01<00:00, 20.22it/s]

Epoch 10/10 | Train Loss: 0.4857 | Train Acc: 0.8290 | Val Loss: 0.3021 | Val Acc: 0.8997 | Time: 11s
Phase 1 training completed.


In [8]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_chicken_phase1_best.pth"),
    device=device,
)

phase1_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase1",
    prefix="chicken_",
)

plot_training_curves(
    phase1_history,
    save_path=FIGURES_DIR / "chicken_training_curves_phase1.png",
)
print("Phase 1 evaluation and plotting completed.")

C:\Users\MangOS\livestock-diseases-ai\src\model.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


predict:   0%|                                          | 0/39 [00:00<?, ?it/s]

predict:   3%|▊                                 | 1/39 [00:14<09:09, 14.46s/it]

predict:   8%|██▌                               | 3/39 [00:14<02:16,  3.79s/it]

predict:  13%|████▎                             | 5/39 [00:14<01:03,  1.87s/it]

...

predict:  82%|███████████████████████████      | 32/39 [00:15<00:00, 12.73it/s]

predict:  90%|█████████████████████████████▌   | 35/39 [00:16<00:00, 14.37it/s]

predict:  97%|████████████████████████████████▏| 38/39 [00:16<00:00, 15.77it/s]


--- Evaluation Results (phase1) ---
Accuracy   : 0.9022
Macro F1   : 0.8536
Weighted F1: 0.9008


Phase 1 evaluation and plotting completed.


In [9]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_chicken_phase1_best.pth"),
    device=device,
)

unfreeze_backbone(model)

params = count_parameters(model)

print("Best Phase 1 checkpoint loaded.")
print("Backbone unfrozen for Phase 2.")
print("Parameters:", params)

Best Phase 1 checkpoint loaded.
Backbone unfrozen for Phase 2.
Parameters: {'total': 11178564, 'trainable': 11178564, 'frozen': 0}


C:\Users\MangOS\livestock-diseases-ai\src\model.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


In [10]:
phase2_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=15,
    learning_rate=0.0001, # LR nhỏ để tinh chỉnh mượt mà
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=str(MODELS_DIR / "resnet18_chicken_phase2_best.pth"),
    history_path=str(RESULTS_DIR / "history_chicken_phase2.json"),
    phase_name="phase2_full_finetuning",
)

print("Phase 2 training completed.")


Training phase2_full_finetuning
Epochs: 15
Learning rate: 0.0001
Trainable parameters: 11,178,564


train:   0%|                                           | 0/177 [00:00<?, ?it/s]

train:   1%|▏                                  | 1/177 [00:00<02:13,  1.32it/s]

train:   1%|▍                                  | 2/177 [00:00<01:11,  2.45it/s]

train:   2%|▌                                  | 3/177 [00:01<00:50,  3.41it/s]

...

eval:  76%|███████████████████████████▍        | 29/38 [00:01<00:00, 20.21it/s]

eval:  84%|██████████████████████████████▎     | 32/38 [00:01<00:00, 20.14it/s]

eval:  92%|█████████████████████████████████▏  | 35/38 [00:01<00:00, 20.21it/s]

eval: 100%|████████████████████████████████████| 38/38 [00:01<00:00, 20.95it/s]

Epoch 15/15 | Train Loss: 0.0193 | Train Acc: 0.9931 | Val Loss: 0.0852 | Val Acc: 0.9743 | Time: 29s
Phase 2 training completed.


In [11]:
load_checkpoint(
    model,
    str(MODELS_DIR / "resnet18_chicken_phase2_best.pth"),
    device=device,
)

phase2_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=RESULTS_DIR,
    figures_dir=FIGURES_DIR,
    label="phase2",
    prefix="chicken_",
)

plot_training_curves(
    phase2_history,
    save_path=FIGURES_DIR / "chicken_training_curves_phase2.png",
)
print("Phase 2 evaluation completed.")

predict:   0%|                                          | 0/39 [00:00<?, ?it/s]

predict:   3%|▊                                 | 1/39 [00:00<00:05,  6.63it/s]

predict:   8%|██▌                               | 3/39 [00:00<00:02, 13.11it/s]

predict:  15%|█████▏                            | 6/39 [00:00<00:01, 16.78it/s]

predict:  23%|███████▊                          | 9/39 [00:00<00:01, 18.32it/s]

...

predict:  77%|█████████████████████████▍       | 30/39 [00:01<00:00, 20.22it/s]

predict:  85%|███████████████████████████▉     | 33/39 [00:01<00:00, 20.26it/s]

predict:  92%|██████████████████████████████▍  | 36/39 [00:01<00:00, 20.31it/s]


--- Evaluation Results (phase2) ---
Accuracy   : 0.9811
Macro F1   : 0.9824
Weighted F1: 0.9811


Phase 2 evaluation completed.
